In [1]:
from constant import openalex_path
import polars as pl
from constant import data_path

In [2]:
preprints = pl.scan_parquet(f"data/preprints_opal_0129_curation_imputed.parquet")

In [3]:
year_lf = pl.scan_parquet(f"{openalex_path}/works/publication_year/*")

stats_curation_df = (
    preprints
    .join(year_lf, on="work_id", how="left")
    .rename({"publication_year":"publication_year_curated"})
    .with_columns(
        (pl.col("publication_year_curated") <= pl.col("published_date").dt.year() +2).alias("2year_curated")
    )
    .group_by(pl.col("published_date").dt.year(), "server")
    .agg(
        pl.col("curated_doi").is_not_null().sum().alias("curated"),
        pl.col("2year_curated").sum().alias("2year_curated"),
        pl.col("doi").len().alias("total"),
    )
    .with_columns(
        (pl.col("curated") / pl.col("total") * 100).alias("curation_ratio"),
        (pl.col("2year_curated") / pl.col("curated") * 100).alias("2year_curation_ratio"),
    )
    .collect(engine="streaming")
)

In [4]:
stats_curation_df

published_date,server,curated,2year_curated,total,curation_ratio,2year_curation_ratio
i32,str,u32,u32,u32,f64,f64
2020,"""medrxiv""",10304,14004,14008,73.557967,135.908385
2020,"""socarxiv""",1074,2241,2244,47.860963,208.659218
1996,"""arxiv""",11728,5708,15861,73.942374,48.66985
2008,"""arxiv""",42389,23706,58741,72.162544,55.924886
2017,"""socarxiv""",663,1164,1164,56.958763,175.565611
…,…,…,…,…,…,…
2019,"""biorxiv""",23158,29140,29147,79.452431,125.831246
2014,"""arxiv""",66265,45960,97648,67.861093,69.357881
2013,"""biorxiv""",83,109,109,76.146789,131.325301


In [5]:
(
    stats_curation_df
    .filter(pl.col("published_date") < 2024 - 2)
    .group_by("server")
    .agg(
        pl.col('curation_ratio').mean(),
        pl.col('2year_curation_ratio').mean(),
    )
)

server,curation_ratio,2year_curation_ratio
str,f64,f64
"""psyarxiv""",50.884608,196.386991
"""medrxiv""",78.733848,127.373931
"""biorxiv""",78.534657,127.331248
"""socarxiv""",48.653203,219.141701
"""arxiv""",69.621267,63.732304
"""osf""",28.721133,445.217716


In [6]:
stats_curation_df.group_by("server").sum().select([
    pl.col("server"),
    pl.col("curated"),
    pl.col("total"),
    pl.col("2year_curated"),
]).with_columns(
    (pl.col("curated") / pl.col("total") * 100).alias("curation_ratio"),
    (pl.col("2year_curated") / pl.col("curated") * 100).alias("2year_curation_ratio"),
).sort("total", descending=True)

server,curated,total,2year_curated,curation_ratio,2year_curation_ratio
str,u32,u32,u32,f64,f64
"""arxiv""",1635472,2634610,1559821,62.076436,95.374363
"""biorxiv""",190450,262926,262783,72.43483,137.980047
"""osf""",11872,70553,69329,16.827066,583.970687
"""medrxiv""",41526,62075,62029,66.896496,149.373886
"""psyarxiv""",18298,39902,39431,45.857351,215.493497
"""socarxiv""",7660,16223,16144,47.216914,210.75718


In [7]:
import numpy as np

(
    stats_curation_df
    .filter(pl.col("publication_year")<=2020)
    .group_by("server")
    .map_groups(
        lambda df: df.select(
            pl.lit(df["server"][0]).alias("server"),
            pl.lit(np.percentile(
                np.random.choice(
                    df["curation_ratio"].to_numpy(),
                    (20000, df.height),
                    replace=True,
                ).mean(axis=1),
                50
            )).alias("median"),
            pl.lit(np.percentile(
                np.random.choice(
                    df["curation_ratio"].to_numpy(),
                    (20000, df.height),
                    replace=True,
                ).mean(axis=1),
                2.5
            )).alias("ci_lower"),
            pl.lit(np.percentile(
                np.random.choice(
                    df["curation_ratio"].to_numpy(),
                    (20000, df.height),
                    replace=True,
                ).mean(axis=1),
                97.5
            )).alias("ci_upper"),
        )
    )
)

ColumnNotFoundError: unable to find column "publication_year"; valid columns: ["published_date", "server", "curated", "2year_curated", "total", "curation_ratio", "2year_curation_ratio"]

In [ ]:
import matplotlib.pyplot as plt
pdf = (
    stats_curation_df
    # preprints
    # .filter(
    #     pl.col("primary_category").is_in(
    #         CNS_citing_preprints_df.unique("primary_category").get_column("primary_category").to_list()
    #     )
    # )
    # .group_by("publication_year_preprint", pl.col("curated_doi").is_null())
    # .agg(pl.len()).sort("publication_year_preprint")
    .pivot(
        on="curated_doi",
        values="len",
        index="publication_year_preprint",
    )
    .fill_null(0)
    .with_columns(
        (pl.col("false") / (pl.col("false") + pl.col("true"))).alias("curation_rate")
    )
    .sort("publication_year_preprint")
)

plt.plot(pdf["publication_year_preprint"], pdf["curation_rate"])
plt.xticks(range(2000, 2025, 5))
plt.ylabel("curation rate")
plt.grid(alpha=0.3)
plt.legend(frameon=False)
plt.tight_layout()

ColumnNotFoundError: unable to find column "curation"; valid columns: ["published_date", "server", "curated", "2year_curated", "total", "curation_ratio", "2year_curation_ratio"]